In [ ]:
!pip -q install -U optuna optuna-integration transformers accelerate peft wandb pyarrow
# peft needs torchao>=0.16 OR none — remove the stale Colab one if present:
!pip -q uninstall -y torchao || true
import os, numpy as np, pandas as pd, torch, optuna, wandb
print("optuna", optuna.__version__, "| CUDA:", torch.cuda.is_available())
wandb.login()
os.environ.setdefault("WANDB_PROJECT", "timesfm-taxi-optuna")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
BEST_DIR    = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/timesfm_optuna_best"
RESOLUTIONS = ["1h"]              # start small; [] = all 9 (slower)
os.makedirs(BEST_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"]); df = df.sort_values(["series_id","split","ts"])
def to_dict(sp): return {s: g.sort_values("ts")["value"].to_numpy(np.float32)
                         for s, g in df[df.split==sp].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
FULL = {s: np.concatenate([TRAIN[s], VAL.get(s, np.array([],np.float32))]) for s in TRAIN}
VAL_START = {s: len(TRAIN[s]) for s in TRAIN}
print("series:", len(TRAIN))


In [ ]:
CONTEXT, HORIZON, EVAL_MAX_WINDOWS = 512, 24, 150
MODEL_ID = "google/timesfm-2.5-200m-transformers"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from transformers import TimesFm2_5ModelForPrediction

def load_base():
    return TimesFm2_5ModelForPrediction.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16 if DEVICE=="cuda" else torch.float32,
        device_map=DEVICE)

def wape(y,p):
    y,p=np.asarray(y,float),np.asarray(p,float); d=np.abs(y).sum()
    return float(np.abs(y-p).sum()/d*100) if d else float("nan")

class RandomWindowDS(torch.utils.data.Dataset):
    def __init__(self, series, C, H, n, seed=42):
        self.s, self.C, self.H = series, C, H
        rng=np.random.default_rng(seed); mn=C+H
        valid=[i for i,x in enumerate(series) if len(x)>=mn]; self.items=[]
        for _ in range(n):
            i=int(rng.choice(valid)); st=int(rng.integers(0,len(series[i])-mn+1)); self.items.append((i,st))
    def __len__(self): return len(self.items)
    def __getitem__(self,k):
        i,st=self.items[k]; x=self.s[i]
        return (torch.tensor(x[st:st+self.C],dtype=torch.float32),
                torch.tensor(x[st+self.C:st+self.C+self.H],dtype=torch.float32))

@torch.no_grad()
def eval_wape(model, batch=128):
    windows=[]
    for sid,arr in FULL.items():
        vs=VAL_START[sid]; origins=[o for o in range(vs,len(arr)-HORIZON+1,HORIZON) if o-CONTEXT>=0]
        if len(origins)>EVAL_MAX_WINDOWS:
            origins=[origins[i] for i in np.linspace(0,len(origins)-1,EVAL_MAX_WINDOWS).astype(int)]
        for o in origins: windows.append((sid,arr[o-CONTEXT:o],arr[o:o+HORIZON]))
    per={}
    for i in range(0,len(windows),batch):
        ch=windows[i:i+batch]
        X=torch.tensor(np.stack([c for _,c,_ in ch]),dtype=torch.float32,device=DEVICE)
        mp=model(past_values=X).mean_predictions[:,:HORIZON].float().cpu().numpy()
        for j,(sid,_,tgt) in enumerate(ch):
            per.setdefault(sid,([],[])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    vals = [wape(np.concatenate(ys), np.concatenate(ps)) for ys, ps in per.values()]
    return float(np.nanmedian(vals)) if vals else float("nan")   # nanmedian skips empty/zero-denom series


In [ ]:
from peft import LoraConfig, get_peft_model

TRAIN_LIST = list(TRAIN.values())
EPOCHS, BATCH, NUM_SAMPLES = 3, 16, 4000     # per-trial budget (raise for final runs)

def objective(trial):
    lr        = trial.suggest_float("lr", 2e-5, 5e-4, log=True)
    lora_r    = trial.suggest_categorical("lora_r", [4, 8, 16, 32])
    lora_alpha= trial.suggest_categorical("lora_alpha", [8, 16, 32, 64])
    wd        = trial.suggest_float("weight_decay", 0.0, 0.1)

    run = wandb.init(project=os.environ["WANDB_PROJECT"], name=f"trial-{trial.number}",
                     config=dict(lr=lr, lora_r=lora_r, lora_alpha=lora_alpha, weight_decay=wd),
                     reinit=True)
    model = get_peft_model(load_base(), LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, target_modules="all-linear",
        lora_dropout=0.05, bias="none")).train()

    ds = RandomWindowDS(TRAIN_LIST, CONTEXT, HORIZON, NUM_SAMPLES)
    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(dl))

    best = float("inf")
    for ep in range(1, EPOCHS+1):
        model.train()
        for ctx, tgt in dl:
            ctx, tgt = ctx.to(DEVICE), tgt.to(DEVICE)
            out = model(past_values=ctx, future_values=tgt, forecast_context_len=CONTEXT)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(); sched.step()
        model.eval()
        w = eval_wape(model)
        if not np.isfinite(w):        # diverged (e.g. too-high LR in bf16) → big penalty, not nan
            w = 1e6
        best = min(best, w)
        wandb.log({"epoch": ep, "val_wape": w})
        trial.report(w, ep)                 # feed the pruner
        if trial.should_prune():
            wandb.finish(); raise optuna.TrialPruned()

    wandb.log({"best_val_wape": best}); wandb.finish()
    # keep the model around only if it's the running best (saved in §5 instead to save disk)
    return best


In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=1)
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner,
                            study_name="timesfm-taxi")
study.optimize(objective, n_trials=15, gc_after_trial=True)   # ~raise for a deeper search

print("BEST WAPE:", round(study.best_value, 3))
print("BEST PARAMS:", study.best_params)


In [ ]:
# Optuna's own plots (hyperparameter importance / history)
try:
    import optuna.visualization as vis
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()
except Exception as e:
    print("viz needs plotly:", e)

# retrain with the best params and save the adapter to Drive
bp = study.best_params
model = get_peft_model(load_base(), LoraConfig(
    r=bp["lora_r"], lora_alpha=bp["lora_alpha"], target_modules="all-linear",
    lora_dropout=0.05, bias="none")).train()
ds = RandomWindowDS(TRAIN_LIST, CONTEXT, HORIZON, NUM_SAMPLES*2)   # a bit longer for the final
dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
opt = torch.optim.AdamW(model.parameters(), lr=bp["lr"], weight_decay=bp["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(dl))
for ep in range(EPOCHS):
    for ctx,tgt in dl:
        ctx,tgt=ctx.to(DEVICE),tgt.to(DEVICE)
        out=model(past_values=ctx, future_values=tgt, forecast_context_len=CONTEXT)
        out.loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step(); opt.zero_grad(); sched.step()
model.eval(); print("final holdout WAPE:", round(eval_wape(model),3))
model.save_pretrained(BEST_DIR); print("saved best adapter →", BEST_DIR)
